In [1]:
# ======================================================================
#  Cell S0 — PREFLIGHT. Validates everything before any training.
#  Reads only. Takes ~30 s. Do not run S1 until this prints READY.
# ======================================================================
import os, shutil, cv2, numpy as np, pandas as pd, torch

ROOT = "/root/autodl-tmp/CBIS"
CSV  = f"{ROOT}/unified_folds_mass_v2.csv"
IMG  = 256
fail, warn = [], []

def chk(cond, msg):
    (fail if not cond else warn).append(msg) if not cond else None
    print(("  ok   " if cond else "  FAIL ") + msg)

print("=" * 68); print("PREFLIGHT"); print("=" * 68)

# ---- 1. manifest ----------------------------------------------------
chk(os.path.exists(CSV), f"{os.path.basename(CSV)} exists")
if not os.path.exists(CSV):
    raise SystemExit("run Cell F2 first")
M = pd.read_csv(CSV)
chk(len(M) == 1696, f"row count = {len(M)} (expect 1696)")
need = ["img", "msk", "lesion_key", "patient_id", "pathology"] + [f"role_f{k}" for k in range(5)]
miss = [c for c in need if c not in M.columns]
chk(not miss, f"required columns present{'' if not miss else ' -- MISSING ' + str(miss)}")
chk(M.lesion_key.nunique() == 1005, f"lesions = {M.lesion_key.nunique()} (expect 1005)")
chk(M.patient_id.nunique() == 892,  f"patients = {M.patient_id.nunique()} (expect 892)")

# ---- 2. crops point at v2 -------------------------------------------
v2 = M.img.astype(str).str.contains("crops_fixed_mass_v2").mean()
chk(v2 == 1.0, f"img column points at v2 crops ({v2:.0%})")

# ---- 3. every file exists and reads ---------------------------------
bad_i = [p for p in M.img if not os.path.exists(p)]
bad_m = [p for p in M.msk if not os.path.exists(p)]
chk(not bad_i, f"all {len(M)} images on disk" + ("" if not bad_i else f" -- {len(bad_i)} missing"))
chk(not bad_m, f"all {len(M)} masks on disk"  + ("" if not bad_m else f" -- {len(bad_m)} missing"))

rng = np.random.default_rng(0)
probe = M.sample(min(120, len(M)), random_state=0)
shapes, empties, nonbin, touch = set(), 0, 0, 0
for _, r in probe.iterrows():
    im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
    mk = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
    if im is None or mk is None:
        empties += 1; continue
    shapes.add(im.shape)
    if im.shape != mk.shape: nonbin += 1
    b = mk > 127
    if not b.any(): empties += 1
    if len(np.unique(mk)) > 8: nonbin += 1
    e = np.concatenate([b[:2].ravel(), b[-2:].ravel(), b[:, :2].ravel(), b[:, -2:].ravel()])
    if e.any(): touch += 1
chk(empties == 0, f"no empty/unreadable masks in {len(probe)} sampled")
chk(nonbin == 0,  f"masks binary and shape-matched in {len(probe)} sampled")
chk(len(shapes) == 1, f"uniform crop size {shapes}")
chk(touch == 0, f"no mask touches the crop edge in {len(probe)} sampled")

# ---- 4. folds and leakage -------------------------------------------
roles = [f"role_f{k}" for k in range(5)]
vals = set()
for c in roles: vals |= set(M[c].astype(str).str.lower().unique())
chk(vals <= {"train", "val", "test"}, f"role values = {sorted(vals)}")

leak = []
for k in range(5):
    r = M[f"role_f{k}"].astype(str).str.lower()
    tr = set(M.loc[r == "train", "patient_id"])
    va = set(M.loc[r == "val",   "patient_id"])
    te = set(M.loc[r == "test",  "patient_id"])
    if tr & va: leak.append(f"f{k} train/val overlap {len(tr & va)}")
    if tr & te: leak.append(f"f{k} train/test overlap {len(tr & te)}")
    if va & te: leak.append(f"f{k} val/test overlap {len(va & te)}")
    print(f"       fold {k}: train {len(tr):3d} / val {len(va):3d} / test {len(te):3d} patients")
chk(not leak, "no patient overlap in any fold" + ("" if not leak else f" -- {leak}"))

ntest = sum((M[f"role_f{k}"].astype(str).str.lower() == "test").sum() for k in range(5))
chk(ntest == len(M), f"each ROI is test exactly once ({ntest} of {len(M)})")

# ---- 5. labels ------------------------------------------------------
y = M.groupby("lesion_key").pathology.first().str.upper().str.contains("MALIGNANT")
chk(0.40 < y.mean() < 0.52, f"malignant prevalence {y.mean():.1%} (expect ~46%)")

# ---- 6. environment -------------------------------------------------
chk(torch.cuda.is_available(), f"CUDA available"
    + (f" -- {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    chk(free / 2**30 > 6, f"GPU free memory {free/2**30:.1f} GB of {total/2**30:.1f} GB")
gb = shutil.disk_usage(ROOT).free / 2**30
chk(gb > 15, f"disk free {gb:.1f} GB (need ~15 for checkpoints + predmasks)")

print("=" * 68)
print("READY — safe to run S1" if not fail else f"NOT READY — {len(fail)} problem(s), fix before training")
print("=" * 68)

PREFLIGHT
  ok   unified_folds_mass_v2.csv exists
  ok   row count = 1696 (expect 1696)
  ok   required columns present
  ok   lesions = 1005 (expect 1005)
  ok   patients = 892 (expect 892)
  ok   img column points at v2 crops (100%)
  ok   all 1696 images on disk
  ok   all 1696 masks on disk
  ok   no empty/unreadable masks in 120 sampled
  ok   masks binary and shape-matched in 120 sampled
  ok   uniform crop size {(512, 512)}
  ok   no mask touches the crop edge in 120 sampled
  ok   role values = ['test', 'train', 'val']
       fold 0: train 629 / val  85 / test 178 patients
       fold 1: train 629 / val  85 / test 178 patients
       fold 2: train 628 / val  85 / test 179 patients
       fold 3: train 631 / val  85 / test 176 patients
       fold 4: train 626 / val  85 / test 181 patients
  ok   no patient overlap in any fold
  ok   each ROI is test exactly once (1696 of 1696)
  ok   malignant prevalence 45.9% (expect ~46%)
  ok   CUDA available -- NVIDIA GeForce RTX 3090
  ok 

In [1]:
# ======================================================================
#  Cell S1 — segmentation on v2 crops, 3 output heads x 5 folds
#
#  Heads:
#    "ds"    standard 2-channel head + deep supervision   (current model)
#    "duel"  original dueling head V + A - mean(A)        (provably degenerate)
#    "duel2" supervised decomposition: logit = A + V,
#            V from a coarser decoder level and supervised on a dilated
#            mask, so the two streams learn region and boundary separately
#
#  Everything else is held identical, so the head is the only variable.
#  Resumable: existing checkpoints are skipped. Safe to re-run.
# ======================================================================
import os, time, json
os.environ["OMP_NUM_THREADS"] = "4"; os.environ["HF_HUB_OFFLINE"] = "1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)

ROOT   = "/root/autodl-tmp/CBIS"
CSV    = f"{ROOT}/unified_folds_mass_v2.csv"
CKPT   = f"{ROOT}/seg_v2"; os.makedirs(CKPT, exist_ok=True)
DEV    = torch.device("cuda")
IMG, BATCH, LR   = 256, 16, 1e-3
EPOCHS, PATIENCE = 60, 10
MULT, THR        = 8, 0.5
TV_A, TV_B       = 0.7, 0.3
DS_W             = [1.0, 0.5, 0.3, 0.2]
LAM_V            = 0.3          # weight of the auxiliary region loss (duel2)
DILATE           = 9            # kernel for the dilated region target
VARIANTS         = ["ds", "duel", "duel2"]
torch.backends.cudnn.benchmark = True

M = pd.read_csv(CSV)
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# ---- cache decoded + CLAHE images once -------------------------------
print("caching images ...", end=" ", flush=True)
CACHE = {}
for p_i, p_m in zip(M["img"], M["msk"]):
    im = cv2.imread(p_i, cv2.IMREAD_GRAYSCALE)
    mk = cv2.imread(p_m, cv2.IMREAD_GRAYSCALE)
    if im.shape != (IMG, IMG): im = cv2.resize(im, (IMG, IMG))
    if mk.shape != (IMG, IMG): mk = cv2.resize(mk, (IMG, IMG), interpolation=cv2.INTER_NEAREST)
    CACHE[p_i] = (_clahe.apply(im), (mk > 127).astype(np.uint8))
print(f"{len(CACHE)} images")

class DS(Dataset):
    def __init__(s, d, aug): s.df = d.reset_index(drop=True); s.aug = aug; s.mult = MULT if aug else 1
    def __len__(s): return len(s.df) * s.mult
    def __getitem__(s, i):
        r = s.df.iloc[i % len(s.df)]; k = i // len(s.df)
        img, m = CACHE[r["img"]]; img = img.copy(); m = m.copy()
        if s.aug and k > 0:
            j = k % 8
            if   j == 1: img, m = np.fliplr(img), np.fliplr(m)
            elif j == 2: img, m = np.flipud(img), np.flipud(m)
            elif j in (3, 4, 5): img, m = np.rot90(img, j - 2), np.rot90(m, j - 2)
            elif j == 6:
                A = cv2.getRotationMatrix2D((IMG / 2, IMG / 2),
                        np.random.uniform(-25, 25), np.random.uniform(.9, 1.1))
                img = cv2.warpAffine(img, A, (IMG, IMG), borderMode=cv2.BORDER_REFLECT)
                m   = cv2.warpAffine(m,   A, (IMG, IMG), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            elif j == 7:
                img = np.clip(img.astype(np.float32) * np.random.uniform(.85, 1.15), 0, 255).astype(np.uint8)
            img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32) / 255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i % len(s.df))

def cb(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class AG(nn.Module):
    def __init__(s, g, x, i):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(g, i, 1), nn.BatchNorm2d(i))
        s.Wx = nn.Sequential(nn.Conv2d(x, i, 1), nn.BatchNorm2d(i))
        s.psi = nn.Sequential(nn.Conv2d(i, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        s.r = nn.ReLU(True)
    def forward(s, g, x): return x * s.psi(s.r(s.Wg(g) + s.Wx(x)))

class ASPP(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        mk = lambda k, d: nn.Sequential(nn.Conv2d(i, o, k, padding=d if k == 3 else 0,
                                                  dilation=d), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b0, s.b1, s.b2, s.b3 = mk(1, 1), mk(3, 6), mk(3, 12), mk(3, 18)
        s.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.proj = nn.Sequential(nn.Conv2d(o * 5, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x):
        g = F.interpolate(s.gp(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return s.proj(torch.cat([s.b0(x), s.b1(x), s.b2(x), s.b3(x), g], 1))

class Net(nn.Module):
    def __init__(s, head="ds", b=32):
        super().__init__()
        s.head = head
        s.e1, s.e2, s.e3, s.e4 = cb(1, b), cb(b, b*2), cb(b*2, b*4), cb(b*4, b*8)
        s.p  = nn.MaxPool2d(2)
        s.bn = ASPP(b*8, b*16)
        s.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2); s.a4 = AG(b*8, b*8, b*4); s.d4 = cb(b*16, b*8)
        s.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2); s.a3 = AG(b*4, b*4, b*2); s.d3 = cb(b*8,  b*4)
        s.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2); s.a2 = AG(b*2, b*2, b);   s.d2 = cb(b*4,  b*2)
        s.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2); s.a1 = AG(b,   b,   b//2);s.d1 = cb(b*2,  b)
        s.ds2, s.ds3, s.ds4 = nn.Conv2d(b*2, 2, 1), nn.Conv2d(b*4, 2, 1), nn.Conv2d(b*8, 2, 1)
        if head == "ds":
            s.out = nn.Conv2d(b, 2, 1)
        elif head == "duel":
            s.val = nn.Conv2d(b, 1, 1); s.adv = nn.Conv2d(b, 2, 1)
        elif head == "duel2":
            s.A = nn.Conv2d(b, 1, 1)        # fine boundary, finest level
            s.V = nn.Conv2d(b*2, 1, 1)      # coarse region, one level up
        else:
            raise ValueError(head)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.p(e1)); e3 = s.e3(s.p(e2)); e4 = s.e4(s.p(e3))
        bo = s.bn(s.p(e4))
        g4 = s.u4(bo); d4 = s.d4(torch.cat([g4, s.a4(g4, e4)], 1))
        g3 = s.u3(d4); d3 = s.d3(torch.cat([g3, s.a3(g3, e3)], 1))
        g2 = s.u2(d3); d2 = s.d2(torch.cat([g2, s.a2(g2, e2)], 1))
        g1 = s.u1(d2); d1 = s.d1(torch.cat([g1, s.a1(g1, e1)], 1))
        vmap = None
        if s.head == "ds":
            main = s.out(d1)
        elif s.head == "duel":
            V, A = s.val(d1), s.adv(d1)
            main = V + A - A.mean(1, keepdim=True)
        else:
            A = s.A(d1)
            vmap = F.interpolate(s.V(d2), size=d1.shape[2:], mode="bilinear", align_corners=False)
            logit = A + vmap
            main = torch.cat([-0.5 * logit, 0.5 * logit], 1)
        if s.training:
            return main, s.ds2(d2), s.ds3(d3), s.ds4(d4), vmap
        return main

def tversky_ce(lo, t):
    ce = F.cross_entropy(lo.float(), t)
    p  = F.softmax(lo.float(), 1)[:, 1]; g = t.float()
    tp = (p * g).sum((1, 2)); fp = (p * (1 - g)).sum((1, 2)); fn = ((1 - p) * g).sum((1, 2))
    tv = (tp + 1) / (tp + TV_A * fp + TV_B * fn + 1)
    return 0.3 * ce + 0.7 * (1 - tv).mean()

def dice_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    return (2 * (p & g).sum() + 1) / (p.sum() + g.sum() + 1)

@torch.no_grad()
def evaluate(net, df):
    net.eval(); out = []
    for _, r in df.iterrows():
        img, m = CACHE[r["img"]]
        x = torch.from_numpy(img.astype(np.float32) / 255.).unsqueeze(0).unsqueeze(0).to(DEV)
        acc = torch.zeros(1, 2, IMG, IMG, device=DEV)
        for k in range(4):                                    # 4-way TTA
            xt = torch.rot90(x, k, (2, 3))
            with torch.amp.autocast("cuda"):
                acc += torch.rot90(net(xt).float(), -k, (2, 3))
        p = (torch.softmax(acc / 4, 1)[0, 1] > THR).cpu().numpy()
        out.append({"lesion_key": r["lesion_key"], "img": r["img"], "dice": dice_np(p, m)})
    return pd.DataFrame(out)

summary = []
for var in VARIANTS:
    for k in range(5):
        tag = f"{var}_fold{k}"
        ck  = f"{CKPT}/seg_{tag}.pth"
        rf  = f"{CKPT}/dice_{tag}.csv"
        if os.path.exists(ck) and os.path.exists(rf):
            d = pd.read_csv(rf)
            print(f"[skip] {tag}  Dice {d.dice.mean():.4f}")
            summary.append({"variant": var, "fold": k, "dice": d.dice.mean(), "n": len(d)})
            continue

        role = M[f"role_f{k}"].astype(str).str.lower()
        tr, va, te = M[role == "train"], M[role == "val"], M[role == "test"]
        assert not (set(tr.patient_id) & set(te.patient_id)), "LEAK"
        assert not (set(tr.patient_id) & set(va.patient_id)), "LEAK"

        torch.manual_seed(1000 + k); np.random.seed(1000 + k)
        net = Net(var).to(DEV)
        opt = torch.optim.Adam(net.parameters(), lr=LR)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "max", patience=4, factor=0.5)
        scaler = torch.amp.GradScaler()
        dl = DataLoader(DS(tr, True), batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)

        best, bad, t0 = -1.0, 0, time.time()
        for ep in range(1, EPOCHS + 1):
            net.train()
            for x, t, _ in dl:
                x, t = x.to(DEV, non_blocking=True), t.to(DEV, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda"):
                    main, s2, s3, s4, vmap = net(x)
                    loss = DS_W[0] * tversky_ce(main, t)
                    for w, s in zip(DS_W[1:], (s2, s3, s4)):
                        td = F.interpolate(t.unsqueeze(1).float(), size=s.shape[2:],
                                           mode="nearest").squeeze(1).long()
                        loss = loss + w * tversky_ce(s, td)
                    if vmap is not None:                       # duel2 region loss
                        dil = F.max_pool2d(t.unsqueeze(1).float(), DILATE, 1, DILATE // 2)
                        loss = loss + LAM_V * F.binary_cross_entropy_with_logits(
                            vmap.float(), dil)
                if not torch.isfinite(loss): continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                scaler.step(opt); scaler.update()

            vd = evaluate(net, va).dice.mean(); sch.step(vd)
            star = ""
            if vd > best:
                best, bad, star = vd, 0, " *"
                torch.save({q: v.cpu() for q, v in net.state_dict().items()}, ck)
            else:
                bad += 1
            print(f"  {tag} ep {ep:2d}/{EPOCHS}  val-Dice {vd:.4f}{star}")
            if bad >= PATIENCE:
                print("  early stop"); break

        net.load_state_dict(torch.load(ck, map_location=DEV)); net.to(DEV)
        res = evaluate(net, te); res.to_csv(rf, index=False)
        print(f"[done] {tag}  TEST Dice {res.dice.mean():.4f} "
              f"(median {res.dice.median():.4f}, n={len(res)})  "
              f"{(time.time()-t0)/60:.0f} min")
        summary.append({"variant": var, "fold": k, "dice": res.dice.mean(), "n": len(res)})

S = pd.DataFrame(summary)
S.to_csv(f"{ROOT}/seg_v2_summary.csv", index=False)
print("\n" + "=" * 68)
print("SEGMENTATION ON v2 CROPS — pooled over 5 patient-grouped folds")
print("=" * 68)
piv = S.groupby("variant").dice.agg(["mean", "std", "count"])
print(piv.to_string())
print("\nper fold:"); print(S.pivot(index="fold", columns="variant", values="dice").to_string())

caching images ... 1696 images
[skip] ds_fold0  Dice 0.9164
[skip] ds_fold1  Dice 0.9202
  ds_fold2 ep  1/60  val-Dice 0.9054 *
  ds_fold2 ep  2/60  val-Dice 0.8977
  ds_fold2 ep  3/60  val-Dice 0.9127 *
  ds_fold2 ep  4/60  val-Dice 0.9129 *
  ds_fold2 ep  5/60  val-Dice 0.9177 *
  ds_fold2 ep  6/60  val-Dice 0.9009
  ds_fold2 ep  7/60  val-Dice 0.9083
  ds_fold2 ep  8/60  val-Dice 0.9202 *
  ds_fold2 ep  9/60  val-Dice 0.9152
  ds_fold2 ep 10/60  val-Dice 0.9155
  ds_fold2 ep 11/60  val-Dice 0.9165
  ds_fold2 ep 12/60  val-Dice 0.9172
  ds_fold2 ep 13/60  val-Dice 0.9170
  ds_fold2 ep 14/60  val-Dice 0.9179
  ds_fold2 ep 15/60  val-Dice 0.9213 *
  ds_fold2 ep 16/60  val-Dice 0.9175
  ds_fold2 ep 17/60  val-Dice 0.9186
  ds_fold2 ep 18/60  val-Dice 0.9194
  ds_fold2 ep 19/60  val-Dice 0.9186
  ds_fold2 ep 20/60  val-Dice 0.9187
  ds_fold2 ep 21/60  val-Dice 0.9220 *
  ds_fold2 ep 22/60  val-Dice 0.9222 *
  ds_fold2 ep 23/60  val-Dice 0.9215
  ds_fold2 ep 24/60  val-Dice 0.9216
  ds_fo

In [1]:
# ======================================================================
#  Cell S2 — generate patient-blind predicted masks from the ds models
#  Each fold predicts ONLY its own test patients. Saves 512px masks and
#  writes unified_folds_mass_v3.csv with a `pmsk` column.
# ======================================================================
import os, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
cv2.setNumThreads(0)

ROOT = "/root/autodl-tmp/CBIS"
CSV  = f"{ROOT}/unified_folds_mass_v2.csv"
CKPT = f"{ROOT}/seg_v2"
OUT  = f"{ROOT}/predmasks_mass_v2"; os.makedirs(OUT, exist_ok=True)
DEV  = torch.device("cuda")
IMG, SAVE, THR = 256, 512, 0.5
VAR = "ds"

def cb(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
class AG(nn.Module):
    def __init__(s, g, x, i):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(g, i, 1), nn.BatchNorm2d(i))
        s.Wx = nn.Sequential(nn.Conv2d(x, i, 1), nn.BatchNorm2d(i))
        s.psi = nn.Sequential(nn.Conv2d(i, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        s.r = nn.ReLU(True)
    def forward(s, g, x): return x * s.psi(s.r(s.Wg(g) + s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        mk = lambda k, d: nn.Sequential(nn.Conv2d(i, o, k, padding=d if k == 3 else 0,
                                                  dilation=d), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b0, s.b1, s.b2, s.b3 = mk(1, 1), mk(3, 6), mk(3, 12), mk(3, 18)
        s.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.proj = nn.Sequential(nn.Conv2d(o * 5, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x):
        g = F.interpolate(s.gp(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return s.proj(torch.cat([s.b0(x), s.b1(x), s.b2(x), s.b3(x), g], 1))
class Net(nn.Module):
    def __init__(s, b=32):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = cb(1, b), cb(b, b*2), cb(b*2, b*4), cb(b*4, b*8)
        s.p, s.bn = nn.MaxPool2d(2), ASPP(b*8, b*16)
        s.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2); s.a4 = AG(b*8, b*8, b*4); s.d4 = cb(b*16, b*8)
        s.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2); s.a3 = AG(b*4, b*4, b*2); s.d3 = cb(b*8,  b*4)
        s.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2); s.a2 = AG(b*2, b*2, b);   s.d2 = cb(b*4,  b*2)
        s.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2); s.a1 = AG(b,   b,   b//2);s.d1 = cb(b*2,  b)
        s.ds2, s.ds3, s.ds4 = nn.Conv2d(b*2, 2, 1), nn.Conv2d(b*4, 2, 1), nn.Conv2d(b*8, 2, 1)
        s.out = nn.Conv2d(b, 2, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.p(e1)); e3 = s.e3(s.p(e2)); e4 = s.e4(s.p(e3))
        bo = s.bn(s.p(e4))
        g4 = s.u4(bo); d4 = s.d4(torch.cat([g4, s.a4(g4, e4)], 1))
        g3 = s.u3(d4); d3 = s.d3(torch.cat([g3, s.a3(g3, e3)], 1))
        g2 = s.u2(d3); d2 = s.d2(torch.cat([g2, s.a2(g2, e2)], 1))
        g1 = s.u1(d2); d1 = s.d1(torch.cat([g1, s.a1(g1, e1)], 1))
        return s.out(d1)

M = pd.read_csv(CSV)
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0].replace("_img", "")

rows = []
for k in range(5):
    ck = f"{CKPT}/seg_{VAR}_fold{k}.pth"
    assert os.path.exists(ck), f"missing {ck}"
    net = Net().to(DEV); net.load_state_dict(torch.load(ck, map_location=DEV)); net.eval()

    te = M[M[f"role_f{k}"].astype(str).str.lower() == "test"]
    tr = set(M.loc[M[f"role_f{k}"].astype(str).str.lower() == "train", "patient_id"])
    assert not (tr & set(te.patient_id)), f"fold {k}: PATIENT LEAK"

    with torch.no_grad():
        for _, r in te.iterrows():
            im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
            if im.shape != (IMG, IMG): im = cv2.resize(im, (IMG, IMG))
            x = torch.from_numpy(_clahe.apply(im).astype(np.float32) / 255.)
            x = x.unsqueeze(0).unsqueeze(0).to(DEV)
            acc = torch.zeros(1, 2, IMG, IMG, device=DEV)
            for t in range(4):
                with torch.amp.autocast("cuda"):
                    acc += torch.rot90(net(torch.rot90(x, t, (2, 3))).float(), -t, (2, 3))
            prob = torch.softmax(acc / 4, 1)[0, 1].cpu().numpy()
            pm = (cv2.resize(prob, (SAVE, SAVE), interpolation=cv2.INTER_LINEAR) > THR)
            p = f"{OUT}/{stem(r['img'])}_pred.png"
            cv2.imwrite(p, pm.astype(np.uint8) * 255)
            rows.append({"img": r["img"], "pmsk": p, "fold_src": k,
                         "pred_cov": float(pm.mean())})
    print(f"fold {k}: {len(te)} masks written")
    del net; torch.cuda.empty_cache()

P = pd.DataFrame(rows)
assert len(P) == len(M) == 1696, f"expected 1696, wrote {len(P)}"
assert P.img.is_unique, "an image was predicted more than once"

N = M.merge(P, on="img", how="left")
assert N.pmsk.notna().all(), "some rows have no predicted mask"
missing = [p for p in N.pmsk if not os.path.exists(p)]
assert not missing, f"{len(missing)} predmask files missing on disk"
N.to_csv(f"{ROOT}/unified_folds_mass_v3.csv", index=False)

empty = (P.pred_cov == 0).sum()
print(f"\nwrote {len(P)} predicted masks -> {OUT}")
print(f"  empty predictions : {empty}")
print(f"  predicted coverage: median {P.pred_cov.median():.1%}  "
      f"min {P.pred_cov.min():.2%}  max {P.pred_cov.max():.1%}")
print(f"  ground-truth cov  : median {M['cov'].median():.1%}")
print(f"\nwrote unified_folds_mass_v3.csv  (adds `pmsk` column)")

fold 0: 339 masks written
fold 1: 340 masks written
fold 2: 339 masks written
fold 3: 339 masks written
fold 4: 339 masks written

wrote 1696 predicted masks -> /root/autodl-tmp/CBIS/predmasks_mass_v2
  empty predictions : 0
  predicted coverage: median 22.3%  min 13.93%  max 26.5%
  ground-truth cov  : median 23.0%

wrote unified_folds_mass_v3.csv  (adds `pmsk` column)


In [4]:
# ======================================================================
#  Cell V3 — inspect the segmentation. Reads only, ~1 minute.
#  Saves a publication-quality figure to figures/seg_v2_qualitative.png
# ======================================================================
import os, cv2, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = "/root/autodl-tmp/CBIS"
FIG  = f"{ROOT}/figures"; os.makedirs(FIG, exist_ok=True)
M = pd.read_csv(f"{ROOT}/unified_folds_mass_v3.csv")

def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    return (2 * (a & b).sum() + 1) / (a.sum() + b.sum() + 1)

rows = []
for _, r in M.iterrows():
    g = cv2.imread(r["msk"],  cv2.IMREAD_GRAYSCALE) > 127
    p = cv2.imread(r["pmsk"], cv2.IMREAD_GRAYSCALE) > 127
    if g.shape != p.shape:
        p = cv2.resize(p.astype(np.uint8), g.shape[::-1], interpolation=cv2.INTER_NEAREST) > 0
    rows.append({"img": r["img"], "msk": r["msk"], "pmsk": r["pmsk"],
                 "lesion_key": r["lesion_key"], "path": r["pathology"],
                 "subtlety": r.get("subtlety"), "shape": r.get("mass_shape"),
                 "dice": dice(p, g), "gt_cov": g.mean(), "pr_cov": p.mean()})
D = pd.DataFrame(rows).sort_values("dice").reset_index(drop=True)
D.to_csv(f"{ROOT}/seg_v2_per_roi.csv", index=False)

print(f"Dice over {len(D)} ROIs")
print(f"  mean {D.dice.mean():.4f}   median {D.dice.median():.4f}   std {D.dice.std():.4f}")
for t in (0.90, 0.85, 0.80, 0.70, 0.50):
    print(f"  >= {t:.2f} : {(D.dice >= t).sum():4d} / {len(D)}  ({(D.dice >= t).mean():.1%})")
print(f"  worst {D.dice.min():.4f}   best {D.dice.max():.4f}")
print(f"\n  size dependence  Spearman(gt coverage, Dice) = "
      f"{D[['gt_cov','dice']].corr(method='spearman').iloc[0,1]:+.3f}")
print(f"  by pathology:\n{D.groupby('path').dice.agg(['mean','count']).to_string()}")
if D.subtlety.notna().any():
    print(f"\n  by subtlety (1 = hardest to see):")
    print(D.groupby('subtlety').dice.agg(['mean', 'count']).to_string())

# ---- pick examples across the whole Dice range -----------------------
n = len(D)
picks = ([("worst", D.index[i]) for i in range(4)] +
         [("median", D.index[i]) for i in range(n // 2 - 2, n // 2 + 2)] +
         [("best", D.index[i]) for i in range(n - 4, n)])

fig, axes = plt.subplots(3, 4, figsize=(13, 10.2))
for ax, (band, idx) in zip(axes.ravel(), picks):
    r = D.loc[idx]
    im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
    g  = (cv2.imread(r["msk"],  cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    p  = (cv2.imread(r["pmsk"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    if p.shape != im.shape: p = cv2.resize(p, im.shape[::-1], interpolation=cv2.INTER_NEAREST)
    if g.shape != im.shape: g = cv2.resize(g, im.shape[::-1], interpolation=cv2.INTER_NEAREST)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    for mask, col in ((g, (0, 220, 0)), (p, (255, 60, 60))):
        cnt, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(rgb, cnt, -1, col, 3)
    ax.imshow(rgb); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{band}   Dice {r['dice']:.3f}\n{str(r['path'])[:22].title()}",
                 fontsize=9)
fig.legend(handles=[Line2D([0], [0], color="green", lw=3, label="radiologist annotation"),
                    Line2D([0], [0], color="red",   lw=3, label="predicted (patient-blind)")],
           loc="lower center", ncol=2, frameon=False, fontsize=11)
fig.suptitle(f"Mass segmentation on held-out patients — mean Dice {D.dice.mean():.3f} "
             f"(n={len(D)})", fontsize=13)
fig.tight_layout(rect=[0, 0.035, 1, 0.97])
fig.savefig(f"{FIG}/seg_v2_qualitative.png", dpi=160)
print(f"\nsaved {FIG}/seg_v2_qualitative.png")

# ---- distribution + agreement ---------------------------------------
fig2, ax = plt.subplots(1, 3, figsize=(14, 3.9))
ax[0].hist(D.dice, bins=45, color="#3a6ea5", edgecolor="white")
ax[0].axvline(D.dice.mean(), color="crimson", ls="--",
              label=f"mean {D.dice.mean():.3f}")
ax[0].set_xlabel("Dice"); ax[0].set_ylabel("regions"); ax[0].legend(frameon=False)
ax[0].set_title("Dice distribution")

ax[1].scatter(D.gt_cov * 100, D.dice, s=6, alpha=.3, color="#3a6ea5")
ax[1].set_xlabel("lesion area (% of crop)"); ax[1].set_ylabel("Dice")
ax[1].set_title("size dependence")

lim = [0, max(D.gt_cov.max(), D.pr_cov.max()) * 100 * 1.05]
ax[2].scatter(D.gt_cov * 100, D.pr_cov * 100, s=6, alpha=.3, color="#3a6ea5")
ax[2].plot(lim, lim, color="crimson", ls="--", lw=1)
ax[2].set_xlim(lim); ax[2].set_ylim(lim)
ax[2].set_xlabel("true area (%)"); ax[2].set_ylabel("predicted area (%)")
ax[2].set_title("area agreement")
fig2.tight_layout(); fig2.savefig(f"{FIG}/seg_v2_distribution.png", dpi=160)
print(f"saved {FIG}/seg_v2_distribution.png")

print("\nworst 10 — inspect these by eye:")
print(D.head(10)[["lesion_key", "path", "dice", "gt_cov", "pr_cov"]].to_string(index=False))

Dice over 1696 ROIs
  mean 0.9208   median 0.9308   std 0.0400
  >= 0.90 : 1299 / 1696  (76.6%)
  >= 0.85 : 1587 / 1696  (93.6%)
  >= 0.80 : 1672 / 1696  (98.6%)
  >= 0.70 : 1693 / 1696  (99.8%)
  >= 0.50 : 1696 / 1696  (100.0%)
  worst 0.6086   best 0.9858

  size dependence  Spearman(gt coverage, Dice) = +0.039
  by pathology:
                             mean  count
path                                    
BENIGN                   0.921543    771
BENIGN_WITHOUT_CALLBACK  0.903037    141
MALIGNANT                0.923206    784

  by subtlety (1 = hardest to see):
              mean  count
subtlety                 
0         0.925734      2
1         0.899905     55
2         0.911588    141
3         0.915827    358
4         0.920006    453
5         0.927398    687

saved /root/autodl-tmp/CBIS/figures/seg_v2_qualitative.png
saved /root/autodl-tmp/CBIS/figures/seg_v2_distribution.png

worst 10 — inspect these by eye:
     lesion_key                    path     dice   gt_cov   pr_co

In [5]:
# ======================================================================
#  Cell V4 — inspect the ten worst regions. Diagnostic only.
# ======================================================================
import cv2, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = "/root/autodl-tmp/CBIS"
D = pd.read_csv(f"{ROOT}/seg_v2_per_roi.csv").sort_values("dice").head(10)

fig, axes = plt.subplots(10, 4, figsize=(11, 27))
for row, (_, r) in enumerate(D.iterrows()):
    im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
    g  = (cv2.imread(r["msk"],  cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    p  = (cv2.imread(r["pmsk"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    if g.shape != im.shape: g = cv2.resize(g, im.shape[::-1], interpolation=cv2.INTER_NEAREST)
    if p.shape != im.shape: p = cv2.resize(p, im.shape[::-1], interpolation=cv2.INTER_NEAREST)

    ov = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    for m, c in ((g, (0, 220, 0)), (p, (255, 60, 60))):
        cnt, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, cnt, -1, c, 3)
    agree = np.zeros((*im.shape, 3), np.uint8)
    agree[..., 1] = (g & p) * 200                       # green  = both
    agree[..., 0] = (p & ~g) * 200                      # red    = predicted only
    agree[..., 2] = (g & ~p) * 200                      # blue   = missed

    for col, (pic, ttl) in enumerate(zip(
            [im, g * 255, p * 255, None],
            ["image", "radiologist", "predicted", "agreement"])):
        ax = axes[row, col]
        ax.imshow(ov if col == 3 else pic, cmap=None if col == 3 else "gray")
        if col == 3: ax.imshow(agree, alpha=.45)
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0: ax.set_title(ttl, fontsize=11)
    axes[row, 0].set_ylabel(f"{r['lesion_key']}\nDice {r['dice']:.3f}\n{str(r['path'])[:14]}",
                            fontsize=7, rotation=0, ha="right", va="center", labelpad=42)
fig.suptitle("Ten lowest-Dice regions — green agree, red over-segmented, blue missed",
             fontsize=13, y=0.999)
fig.tight_layout(rect=[0, 0, 1, 0.993])
fig.savefig(f"{ROOT}/figures/seg_v2_worst10.png", dpi=140)
print(f"saved {ROOT}/figures/seg_v2_worst10.png")
print(D[["lesion_key", "path", "dice", "gt_cov", "pr_cov", "subtlety", "shape"]].to_string(index=False))

saved /root/autodl-tmp/CBIS/figures/seg_v2_worst10.png
     lesion_key                    path     dice   gt_cov   pr_cov  subtlety                              shape
P_00778_RIGHT_2                  BENIGN 0.608577 0.226826 0.259277         3           ARCHITECTURAL_DISTORTION
P_00173_RIGHT_1                  BENIGN 0.652276 0.231728 0.209972         5                               OVAL
P_00778_RIGHT_1                  BENIGN 0.682582 0.136890 0.222561         3           ARCHITECTURAL_DISTORTION
P_00169_RIGHT_1 BENIGN_WITHOUT_CALLBACK 0.707790 0.229275 0.191708         5                          IRREGULAR
 P_00719_LEFT_1               MALIGNANT 0.754047 0.230789 0.229214         5                          IRREGULAR
P_01776_RIGHT_1                  BENIGN 0.770063 0.126617 0.177692         3                          IRREGULAR
P_00116_RIGHT_1               MALIGNANT 0.774559 0.229671 0.220543         5 IRREGULAR-ARCHITECTURAL_DISTORTION
 P_00482_LEFT_1                  BENIGN 0.774651 

In [6]:
# ======================================================================
#  Cell F3 — crops that never invent pixels. Slide, then shrink.
# ======================================================================
import os, glob, cv2, numpy as np, pandas as pd
ROOT, OUT, TARGET, SIDE = "/root/autodl-tmp/CBIS", None, 0.23, 512
OUT = f"{ROOT}/crops_fixed_mass_v3"; os.makedirs(OUT, exist_ok=True)

def sfiles(s): return sorted(glob.glob(f"{ROOT}/jpeg/{s}/*.jpg") + glob.glob(f"{ROOT}/jpeg/{s}/*.png"))
def pick_mask(s):
    best, nb = None, 10**9
    for f in sfiles(s):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        n = len(np.unique(im[::8, ::8]))
        if n < nb: best, nb = im, n
    return best

M = pd.read_csv(f"{ROOT}/unified_folds_mass.csv")
rows, slid, shrunk = [], 0, 0
for _, r in M.iterrows():
    fm, fs = pick_mask(r["mask_series"]), sfiles(r["full_series"])
    fi = cv2.imread(fs[0], cv2.IMREAD_GRAYSCALE)
    if fm.shape != fi.shape:
        fm = cv2.resize(fm, fi.shape[::-1], interpolation=cv2.INTER_NEAREST)
    b = (fm > 127).astype(np.uint8)
    H, W = fi.shape
    ys, xs = np.where(b); y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    cy, cx = (y0 + y1) / 2.0, (x0 + x1) / 2.0
    bh, bw = y1 - y0 + 1, x1 - x0 + 1

    side = max(np.sqrt(b.sum() / TARGET), 1.15 * max(bh, bw))
    side = min(side, H, W)                                   # never exceed the image
    if side < max(bh, bw): side = min(max(bh, bw), H, W); shrunk += 1
    half = side / 2.0
    ncy = float(np.clip(cy, half, H - half))                 # slide inward
    ncx = float(np.clip(cx, half, W - half))
    if abs(ncy - cy) > 1 or abs(ncx - cx) > 1: slid += 1
    Y0, X0 = int(round(ncy - half)), int(round(ncx - half))
    Y1, X1 = Y0 + int(round(side)), X0 + int(round(side))
    Y0, X0 = max(0, Y0), max(0, X0); Y1, X1 = min(H, Y1), min(W, X1)

    ci = cv2.resize(fi[Y0:Y1, X0:X1], (SIDE, SIDE), interpolation=cv2.INTER_AREA)
    cm = cv2.resize((b * 255)[Y0:Y1, X0:X1], (SIDE, SIDE), interpolation=cv2.INTER_NEAREST)
    st = os.path.splitext(os.path.basename(r["img"]))[0].replace("_img", "")
    pi, pm = f"{OUT}/{st}_img.png", f"{OUT}/{st}_msk.png"
    cv2.imwrite(pi, ci); cv2.imwrite(pm, cm)
    bb = cm > 127
    e = np.concatenate([bb[:2].ravel(), bb[-2:].ravel(), bb[:, :2].ravel(), bb[:, -2:].ravel()])
    rows.append({"lesion_key": r["lesion_key"], "img_v3": pi, "msk_v3": pm,
                 "cov": float(bb.mean()), "touch": bool(e.any()),
                 "kept": float(bb.sum()) / max(b.sum() * (SIDE * SIDE) / max((Y1-Y0)*(X1-X0), 1), 1)})

V = pd.DataFrame(rows); V.to_csv(f"{ROOT}/crops_v3_manifest.csv", index=False)
print(f"regenerated {len(V)}   slid {slid}   shrunk {shrunk}   NO synthetic pixels")
print(f"  edge-touching : {V.touch.sum()}  (nonzero is expected where the lesion "
      f"reaches the mammogram border)")
print(f"  coverage      : median {V['cov'].median():.1%}  min {V['cov'].min():.2%}  "
      f"max {V['cov'].max():.1%}")

# rebuild the manifest
N = pd.read_csv(f"{ROOT}/unified_folds_mass.csv")
st = lambda p: os.path.splitext(os.path.basename(str(p)))[0].replace("_img", "")
N["_s"] = N["img"].map(st); V["_s"] = V["img_v3"].map(st)
N = N.merge(V[["_s", "img_v3", "msk_v3", "cov", "touch"]], on="_s", how="left").drop(columns=["_s"])
assert N.img_v3.notna().all()
N["img"], N["msk"] = N["img_v3"], N["msk_v3"]
N = N.drop(columns=["img_v3", "msk_v3"])
N.to_csv(f"{ROOT}/unified_folds_mass_v4.csv", index=False)
print(f"wrote unified_folds_mass_v4.csv ({len(N)} rows)")

regenerated 1696   slid 153   shrunk 0   NO synthetic pixels
  edge-touching : 58  (nonzero is expected where the lesion reaches the mammogram border)
  coverage      : median 23.0%  min 12.10%  max 23.1%
wrote unified_folds_mass_v4.csv (1696 rows)


In [1]:
# ======================================================================
#  Cell S3 — SELF-CONTAINED. Retrain segmentation on v4 crops,
#  write predmasks, write unified_folds_mass_v5.csv. Resumable.
# ======================================================================
import os, time
os.environ["OMP_NUM_THREADS"] = "4"; os.environ["HF_HUB_OFFLINE"] = "1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)

ROOT = "/root/autodl-tmp/CBIS"
CSV  = f"{ROOT}/unified_folds_mass_v4.csv"
CK   = f"{ROOT}/seg_v3";          os.makedirs(CK, exist_ok=True)
PM   = f"{ROOT}/predmasks_mass_v3"; os.makedirs(PM, exist_ok=True)
DEV  = torch.device("cuda")
IMG, SAVE, THR   = 256, 512, 0.5
BATCH, LR        = 16, 1e-3
EPOCHS, PATIENCE = 60, 10
MULT, TV_A, TV_B = 8, 0.7, 0.3
DS_W = [1.0, 0.5, 0.3, 0.2]
torch.backends.cudnn.benchmark = True

M = pd.read_csv(CSV)
assert len(M) == 1696 and M.img.is_unique
_cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
print("caching ...", end=" ", flush=True)
CACHE = {}
for pi, pm in zip(M["img"], M["msk"]):
    im = cv2.imread(pi, cv2.IMREAD_GRAYSCALE); mk = cv2.imread(pm, cv2.IMREAD_GRAYSCALE)
    assert im is not None and mk is not None, pi
    if im.shape != (IMG, IMG): im = cv2.resize(im, (IMG, IMG))
    if mk.shape != (IMG, IMG): mk = cv2.resize(mk, (IMG, IMG), interpolation=cv2.INTER_NEAREST)
    CACHE[pi] = (_cl.apply(im), (mk > 127).astype(np.uint8))
print(len(CACHE))

class SegDS(Dataset):
    def __init__(s, d, aug): s.df = d.reset_index(drop=True); s.aug = aug; s.mult = MULT if aug else 1
    def __len__(s): return len(s.df) * s.mult
    def __getitem__(s, i):
        r = s.df.iloc[i % len(s.df)]; k = i // len(s.df)
        img, m = CACHE[r["img"]]; img = img.copy(); m = m.copy()
        if s.aug and k > 0:
            j = k % 8
            if   j == 1: img, m = np.fliplr(img), np.fliplr(m)
            elif j == 2: img, m = np.flipud(img), np.flipud(m)
            elif j in (3, 4, 5): img, m = np.rot90(img, j - 2), np.rot90(m, j - 2)
            elif j == 6:
                A = cv2.getRotationMatrix2D((IMG/2, IMG/2),
                        np.random.uniform(-25, 25), np.random.uniform(.9, 1.1))
                img = cv2.warpAffine(np.ascontiguousarray(img), A, (IMG, IMG),
                                     borderMode=cv2.BORDER_REFLECT)
                m   = cv2.warpAffine(np.ascontiguousarray(m), A, (IMG, IMG),
                                     flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
            elif j == 7:
                img = np.clip(img.astype(np.float32) * np.random.uniform(.85, 1.15),
                              0, 255).astype(np.uint8)
            img, m = np.ascontiguousarray(img), np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32) / 255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i % len(s.df))

def cblk(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class AttnGate(nn.Module):
    def __init__(s, g, x, i):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(g, i, 1), nn.BatchNorm2d(i))
        s.Wx = nn.Sequential(nn.Conv2d(x, i, 1), nn.BatchNorm2d(i))
        s.psi = nn.Sequential(nn.Conv2d(i, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        s.r = nn.ReLU(True)
    def forward(s, g, x): return x * s.psi(s.r(s.Wg(g) + s.Wx(x)))

class ASPPBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        mk = lambda k, d: nn.Sequential(nn.Conv2d(i, o, k, padding=d if k == 3 else 0,
                                                  dilation=d), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b0, s.b1, s.b2, s.b3 = mk(1, 1), mk(3, 6), mk(3, 12), mk(3, 18)
        s.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i, o, 1),
                             nn.BatchNorm2d(o), nn.ReLU(True))
        s.proj = nn.Sequential(nn.Conv2d(o*5, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x):
        g = F.interpolate(s.gp(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return s.proj(torch.cat([s.b0(x), s.b1(x), s.b2(x), s.b3(x), g], 1))

class SegNet(nn.Module):
    def __init__(s, b=32):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = cblk(1, b), cblk(b, b*2), cblk(b*2, b*4), cblk(b*4, b*8)
        s.p, s.bn = nn.MaxPool2d(2), ASPPBlock(b*8, b*16)
        s.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2); s.a4 = AttnGate(b*8, b*8, b*4); s.d4 = cblk(b*16, b*8)
        s.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2); s.a3 = AttnGate(b*4, b*4, b*2); s.d3 = cblk(b*8,  b*4)
        s.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2); s.a2 = AttnGate(b*2, b*2, b);   s.d2 = cblk(b*4,  b*2)
        s.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2); s.a1 = AttnGate(b,   b,   b//2);s.d1 = cblk(b*2,  b)
        s.out = nn.Conv2d(b, 2, 1)
        s.ds2, s.ds3, s.ds4 = nn.Conv2d(b*2, 2, 1), nn.Conv2d(b*4, 2, 1), nn.Conv2d(b*8, 2, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.p(e1)); e3 = s.e3(s.p(e2)); e4 = s.e4(s.p(e3))
        bo = s.bn(s.p(e4))
        g4 = s.u4(bo); d4 = s.d4(torch.cat([g4, s.a4(g4, e4)], 1))
        g3 = s.u3(d4); d3 = s.d3(torch.cat([g3, s.a3(g3, e3)], 1))
        g2 = s.u2(d3); d2 = s.d2(torch.cat([g2, s.a2(g2, e2)], 1))
        g1 = s.u1(d2); d1 = s.d1(torch.cat([g1, s.a1(g1, e1)], 1))
        if s.training: return s.out(d1), s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return s.out(d1)

def tversky_ce(lo, t):
    ce = F.cross_entropy(lo.float(), t)
    p = F.softmax(lo.float(), 1)[:, 1]; g = t.float()
    tp = (p*g).sum((1, 2)); fp = (p*(1-g)).sum((1, 2)); fn = ((1-p)*g).sum((1, 2))
    return 0.3*ce + 0.7*(1 - (tp+1)/(tp + TV_A*fp + TV_B*fn + 1)).mean()

def dice_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    return (2*(p & g).sum() + 1) / (p.sum() + g.sum() + 1)

@torch.no_grad()
def evaluate(net, df):
    net.eval(); out = []
    for _, r in df.iterrows():
        img, m = CACHE[r["img"]]
        x = torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
        acc = torch.zeros(1, 2, IMG, IMG, device=DEV)
        for t in range(4):
            with torch.amp.autocast("cuda"):
                acc += torch.rot90(net(torch.rot90(x, t, (2, 3))).float(), -t, (2, 3))
        out.append({"img": r["img"], "lesion_key": r["lesion_key"],
                    "dice": dice_np((torch.softmax(acc/4, 1)[0, 1] > THR).cpu().numpy(), m)})
    return pd.DataFrame(out)

stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0].replace("_img", "")
rows = []
for k in range(5):
    ck = f"{CK}/seg_ds_fold{k}.pth"
    role = M[f"role_f{k}"].astype(str).str.lower()
    tr, va, te = M[role == "train"], M[role == "val"], M[role == "test"]
    assert not (set(tr.patient_id) & set(te.patient_id)), f"fold {k} LEAK"
    assert not (set(tr.patient_id) & set(va.patient_id)), f"fold {k} LEAK"

    if not os.path.exists(ck):
        torch.manual_seed(1000+k); np.random.seed(1000+k)
        net = SegNet().to(DEV)
        opt = torch.optim.Adam(net.parameters(), lr=LR)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "max", patience=4, factor=0.5)
        sc  = torch.amp.GradScaler()
        dl  = DataLoader(SegDS(tr, True), batch_size=BATCH, shuffle=True,
                         num_workers=0, drop_last=True)
        best, bad, t0 = -1.0, 0, time.time()
        for ep in range(1, EPOCHS+1):
            net.train()
            for x, t, _ in dl:
                x, t = x.to(DEV), t.to(DEV); opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda"):
                    main, s2, s3, s4 = net(x)
                    loss = DS_W[0]*tversky_ce(main, t)
                    for w, s in zip(DS_W[1:], (s2, s3, s4)):
                        td = F.interpolate(t.unsqueeze(1).float(), size=s.shape[2:],
                                           mode="nearest").squeeze(1).long()
                        loss = loss + w*tversky_ce(s, td)
                if not torch.isfinite(loss): continue
                sc.scale(loss).backward(); sc.unscale_(opt)
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); sc.step(opt); sc.update()
            vd = evaluate(net, va).dice.mean(); sch.step(vd)
            if vd > best:
                best, bad = vd, 0
                torch.save({q: v.cpu() for q, v in net.state_dict().items()}, ck)
            else: bad += 1
            print(f"  f{k} ep {ep:2d}/{EPOCHS} val-Dice {vd:.4f}{' *' if bad == 0 else ''}")
            if bad >= PATIENCE: print("  early stop"); break
        print(f"  fold {k} trained in {(time.time()-t0)/60:.0f} min")
        del net; torch.cuda.empty_cache()

    net = SegNet().to(DEV); net.load_state_dict(torch.load(ck, map_location=DEV)); net.eval()
    res = evaluate(net, te); res.to_csv(f"{CK}/dice_ds_fold{k}.csv", index=False)
    print(f"  fold {k} TEST Dice {res.dice.mean():.4f} (n={len(res)})")
    with torch.no_grad():
        for _, r in te.iterrows():
            img, _ = CACHE[r["img"]]
            x = torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            acc = torch.zeros(1, 2, IMG, IMG, device=DEV)
            for t in range(4):
                with torch.amp.autocast("cuda"):
                    acc += torch.rot90(net(torch.rot90(x, t, (2, 3))).float(), -t, (2, 3))
            prob = torch.softmax(acc/4, 1)[0, 1].cpu().numpy()
            pmk = cv2.resize(prob, (SAVE, SAVE), interpolation=cv2.INTER_LINEAR) > THR
            pth = f"{PM}/{stem(r['img'])}_pred.png"
            cv2.imwrite(pth, pmk.astype(np.uint8)*255)
            rows.append({"img": r["img"], "pmsk": pth, "pred_cov": float(pmk.mean())})
    del net; torch.cuda.empty_cache()

P = pd.DataFrame(rows)
assert len(P) == 1696 and P.img.is_unique, f"got {len(P)}"
N = M.merge(P, on="img", how="left"); assert N.pmsk.notna().all()
assert all(os.path.exists(p) for p in N.pmsk)
N.to_csv(f"{ROOT}/unified_folds_mass_v5.csv", index=False)
d = pd.concat([pd.read_csv(f"{CK}/dice_ds_fold{k}.csv") for k in range(5)])
print(f"\nDICE {d.dice.mean():.4f} over {len(d)} ROIs")
print(f"masks {len(P)}  empty {(P.pred_cov == 0).sum()}  cov median {P.pred_cov.median():.1%}")
print("wrote unified_folds_mass_v5.csv")

caching ... 1696
  fold 0 TEST Dice 0.9018 (n=339)
  fold 1 TEST Dice 0.9059 (n=340)
  fold 2 TEST Dice 0.9048 (n=339)
  fold 3 TEST Dice 0.9107 (n=339)
  fold 4 TEST Dice 0.9079 (n=339)

DICE 0.9062 over 1696 ROIs
masks 1696  empty 0  cov median 22.3%
wrote unified_folds_mass_v5.csv


In [6]:
# ======================================================================
#  Cell C1 — SELF-CONTAINED. Mask-guided dual pooling on v4 crops.
#  guided = dual pooling with predicted mask | blind = no mask control
#  5 folds x 2 seeds. Guided first. Resumable.
# ======================================================================
import os, time, warnings
os.environ["OMP_NUM_THREADS"] = "4"; os.environ["HF_HUB_OFFLINE"] = "1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision.models import densenet121
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore"); cv2.setNumThreads(0)

ROOT = "/root/autodl-tmp/CBIS"
CSV  = f"{ROOT}/unified_folds_mass_v5.csv"
CKPT = f"{ROOT}/cls_v3"; os.makedirs(CKPT, exist_ok=True)
DEV  = torch.device("cuda")
SIZE, BATCH    = 512, 12
EPOCHS, WARM   = 20, 4
LR_HEAD, LR_BB = 3e-4, 3e-5
WD, GAMMA, AUX_W = 1e-3, 1.0, 0.03
SEEDS, VARIANTS  = [11, 22], ["guided", "blind"]
torch.backends.cudnn.benchmark = True

M = pd.read_csv(CSV)
assert {"img", "pmsk", "lesion_key", "patient_id", "pathology"} <= set(M.columns)
assert len(M) == 1696 and M.img.is_unique
M["y"] = M.pathology.astype(str).str.upper().str.contains("MALIGNANT").astype(int)

AUX = {}
for col, name in [("subtlety", "sub"), ("mass_shape", "shp"), ("mass_margins", "mrg")]:
    v = M[col].astype(str).str.strip().str.upper()
    v = v.where(~v.isin(["NAN", "NONE", ""]), other=None)
    cats = sorted(v.dropna().unique())
    M[f"aux_{name}"] = v.map({c: i for i, c in enumerate(cats)}).fillna(-1).astype(int)
    AUX[name] = len(cats)
    print(f"aux {name}: {len(cats)} classes, {(M[f'aux_{name}'] < 0).sum()} missing")

print("caching ...", end=" ", flush=True)
CACHE = {}
for pi, pm in zip(M["img"], M["pmsk"]):
    im = cv2.imread(pi, cv2.IMREAD_GRAYSCALE); mk = cv2.imread(pm, cv2.IMREAD_GRAYSCALE)
    assert im is not None and mk is not None, pi
    if im.shape != (SIZE, SIZE): im = cv2.resize(im, (SIZE, SIZE))
    if mk.shape != (SIZE, SIZE): mk = cv2.resize(mk, (SIZE, SIZE), interpolation=cv2.INTER_NEAREST)
    CACHE[pi] = (im, (mk > 127).astype(np.uint8))
print(len(CACHE))
MEAN, STD = 0.449, 0.226

class ClsDS(Dataset):
    def __init__(s, d, aug): s.df = d.reset_index(drop=True); s.aug = aug
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        im, mk = CACHE[r["img"]]; im = im.copy(); mk = mk.copy()
        if s.aug:
            if np.random.rand() < .5: im, mk = np.fliplr(im), np.fliplr(mk)
            if np.random.rand() < .5: im, mk = np.flipud(im), np.flipud(mk)
            k = np.random.randint(4)
            if k: im, mk = np.rot90(im, k), np.rot90(mk, k)
            if np.random.rand() < .5:
                A = cv2.getRotationMatrix2D((SIZE/2, SIZE/2),
                        np.random.uniform(-20, 20), np.random.uniform(.9, 1.1))
                im = cv2.warpAffine(np.ascontiguousarray(im), A, (SIZE, SIZE),
                                    borderMode=cv2.BORDER_REFLECT)
                mk = cv2.warpAffine(np.ascontiguousarray(mk), A, (SIZE, SIZE),
                                    flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < .5:
                im = np.clip(im.astype(np.float32)*np.random.uniform(.85, 1.15),
                             0, 255).astype(np.uint8)
            im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
        x = (im.astype(np.float32)/255. - MEAN)/STD
        return (torch.from_numpy(x).unsqueeze(0),
                torch.from_numpy(mk.astype(np.float32)).unsqueeze(0),
                torch.tensor(float(r["y"])),
                torch.tensor([r["aux_sub"], r["aux_shp"], r["aux_mrg"]]), i)

class ClsNet(nn.Module):
    def __init__(s, guided=True):
        super().__init__()
        s.guided = guided
        try:    bb = densenet121(weights="IMAGENET1K_V1")
        except TypeError: bb = densenet121(pretrained=True)
        w = bb.features.conv0.weight.data.sum(1, keepdim=True)
        bb.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        bb.features.conv0.weight.data = w
        s.bb = bb.features
        d = 2048 if guided else 1024
        s.drop = nn.Dropout(0.3); s.head = nn.Linear(d, 1)
        s.aux = nn.ModuleDict({k: nn.Linear(d, n) for k, n in AUX.items()})
    def forward(s, x, m):
        f = F.relu(s.bb(x), inplace=True)
        f_u = f.mean((2, 3))
        if s.guided:
            md = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
            w = 1.0 + 2.0*md
            f_g = (f*w).sum((2, 3)) / w.sum((2, 3)).clamp(min=1e-6)
            z = torch.cat([f_g, f_u], 1)
        else:
            z = f_u
        z = s.drop(z)
        return s.head(z).squeeze(1), {k: s.aux[k](z) for k in s.aux}

def focal(logit, y, alpha, gamma=GAMMA):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, y, reduction="none")
    pt = p*y + (1-p)*(1-y); at = alpha*y + (1-alpha)*(1-y)
    return (at*(1-pt).pow(gamma)*ce).mean()

@torch.no_grad()
def predict(net, df):
    net.eval(); ids, ps = [], []
    for x, m, y, a, idx in DataLoader(ClsDS(df, False), batch_size=BATCH,
                                      shuffle=False, num_workers=0):
        x, m = x.to(DEV), m.to(DEV); acc = 0
        for k in range(4):
            with torch.amp.autocast("cuda"):
                lo, _ = net(torch.rot90(x, k, (2, 3)), torch.rot90(m, k, (2, 3)))
            acc = acc + torch.sigmoid(lo.float())
        ids.append(idx.numpy()); ps.append((acc/4).cpu().numpy())
    i = np.concatenate(ids); p = np.concatenate(ps)
    out = df.reset_index(drop=True).iloc[i].copy(); out["prob"] = p
    return out

def lesion_auc(df):
    g = df.groupby("lesion_key").agg(yy=("y", "max"), pp=("prob", "mean"))
    return roc_auc_score(g.yy, g.pp)

for var in VARIANTS:
    oof = []
    for k in range(5):
        role = M[f"role_f{k}"].astype(str).str.lower()
        tr, va, te = M[role == "train"], M[role == "val"], M[role == "test"]
        assert not (set(tr.patient_id) & set(te.patient_id)), f"f{k} LEAK"
        assert not (set(tr.patient_id) & set(va.patient_id)), f"f{k} LEAK"
        assert not (set(va.patient_id) & set(te.patient_id)), f"f{k} LEAK"
        alpha = float(1.0 - tr.y.mean())

        sp = []
        for sd in SEEDS:
            tag = f"{var}_f{k}_s{sd}"; pf = f"{CKPT}/pred_{tag}.csv"
            if os.path.exists(pf):
                sp.append(pd.read_csv(pf)); print(f"[skip] {tag}"); continue
            torch.manual_seed(sd+k); np.random.seed(sd+k)
            net = ClsNet(var == "guided").to(DEV)
            for p in net.bb.parameters(): p.requires_grad = False
            opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
                                    lr=LR_HEAD, weight_decay=WD)
            sch, sc = None, torch.amp.GradScaler()
            dl = DataLoader(ClsDS(tr, True), batch_size=BATCH, shuffle=True,
                            num_workers=0, drop_last=True)
            best, bp, t0 = -1.0, f"{CKPT}/net_{tag}.pth", time.time()
            for ep in range(1, EPOCHS+1):
                if ep == WARM+1:
                    for p in net.bb.parameters(): p.requires_grad = True
                    opt = torch.optim.AdamW([
                        {"params": net.bb.parameters(), "lr": LR_BB},
                        {"params": [p for n, p in net.named_parameters()
                                    if not n.startswith("bb.")], "lr": LR_HEAD}],
                        weight_decay=WD)
                    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS-WARM, 1))
                net.train()
                for x, m, y, a, _ in dl:
                    x, m, y, a = x.to(DEV), m.to(DEV), y.to(DEV), a.to(DEV)
                    opt.zero_grad(set_to_none=True)
                    with torch.amp.autocast("cuda"):
                        lo, aux = net(x, m)
                        loss = focal(lo.float(), y, alpha)
                        for j, key in enumerate(("sub", "shp", "mrg")):
                            loss = loss + AUX_W*F.cross_entropy(aux[key].float(),
                                                                a[:, j], ignore_index=-1)
                    if not torch.isfinite(loss): continue
                    sc.scale(loss).backward(); sc.unscale_(opt)
                    nn.utils.clip_grad_norm_(net.parameters(), 5.0); sc.step(opt); sc.update()
                if sch: sch.step()
                vauc = lesion_auc(predict(net, va))
                if vauc > best:
                    best = vauc
                    torch.save({q: v.cpu() for q, v in net.state_dict().items()}, bp)
                print(f"  {tag} ep {ep:2d}/{EPOCHS} val-AUC {vauc:.4f}"
                      f"{' *' if vauc == best else ''}")
            net.load_state_dict(torch.load(bp, map_location=DEV)); net.to(DEV)
            pr = predict(net, te)[["img", "lesion_key", "y", "prob"]]
            pr.to_csv(pf, index=False); sp.append(pr)
            print(f"[done] {tag} test lesion-AUC {lesion_auc(pr):.4f} "
                  f"{(time.time()-t0)/60:.0f} min")
            del net; torch.cuda.empty_cache()

        ss = [q.sort_values("img").reset_index(drop=True) for q in sp]
        assert all((q.img.values == ss[0].img.values).all() for q in ss), "seed mismatch"
        fold = ss[0][["img", "lesion_key", "y"]].copy()
        fold["prob"] = np.mean([q["prob"].values for q in ss], axis=0)
        oof.append(fold)
        print(f"  fold {k} ({var}) seed-averaged lesion-AUC {lesion_auc(fold):.4f}")

    O = pd.concat(oof, ignore_index=True)
    assert len(O) == 1696 and O.img.is_unique
    g = O.groupby("lesion_key").agg(yy=("y", "max"), pp=("prob", "mean"))
    auc = roc_auc_score(g.yy, g.pp)
    O.rename(columns={"y": "true"}).to_csv(f"{ROOT}/cv_mass_{var}_v4_oof.csv", index=False)
    print(f"\n=== {var.upper()} lesion-level AUC {auc:.4f} (n={len(g)}) -> "
          f"cv_mass_{var}_v4_oof.csv\n")

aux sub: 6 classes, 0 missing
aux shp: 20 classes, 4 missing
aux mrg: 19 classes, 60 missing
caching ... 

1696
  guided_f0_s11 ep  1/20 val-AUC 0.5207 *
  guided_f0_s11 ep  2/20 val-AUC 0.5942 *
  guided_f0_s11 ep  3/20 val-AUC 0.5966 *
  guided_f0_s11 ep  4/20 val-AUC 0.6135 *
  guided_f0_s11 ep  5/20 val-AUC 0.6942 *
  guided_f0_s11 ep  6/20 val-AUC 0.6942
  guided_f0_s11 ep  7/20 val-AUC 0.7620 *
  guided_f0_s11 ep  8/20 val-AUC 0.8019 *
  guided_f0_s11 ep  9/20 val-AUC 0.7245
  guided_f0_s11 ep 10/20 val-AUC 0.8043 *
  guided_f0_s11 ep 11/20 val-AUC 0.7923
  guided_f0_s11 ep 12/20 val-AUC 0.8120 *
  guided_f0_s11 ep 13/20 val-AUC 0.8024
  guided_f0_s11 ep 14/20 val-AUC 0.8168 *
  guided_f0_s11 ep 15/20 val-AUC 0.8163
  guided_f0_s11 ep 16/20 val-AUC 0.8159
  guided_f0_s11 ep 17/20 val-AUC 0.8125
  guided_f0_s11 ep 18/20 val-AUC 0.8087
  guided_f0_s11 ep 19/20 val-AUC 0.8082
  guided_f0_s11 ep 20/20 val-AUC 0.8053
[done] guided_f0_s11 test lesion-AUC 0.8240 6 min
  guided_f0_s22 ep  1/20 val-AUC 0.6192 *
  guided_f0_s22 ep  2/20 val-AUC 0.6202 *
  guided_f0_s22 ep  3/20 val-AUC 0.6111
 